# Thực hành Seaborn từ cơ bản đến ứng dụng

**Ngôn ngữ:** Python 3  
**Thư viện chính:** `seaborn`, `matplotlib`, `pandas`, `numpy`  
**Định hướng ứng dụng:** Data Science, Business Analytics, Economics

Notebook này được thiết kế theo cấu trúc:
**khái niệm → ví dụ → bài tập nhỏ → ứng dụng → bài tổng hợp → bài tự làm**.

> Các bộ dữ liệu trong notebook được tạo mô phỏng để có thể chạy độc lập, không cần Internet.

## Mục tiêu học tập

Sau bài thực hành, người học có thể:

- hiểu Seaborn xây dựng trên Matplotlib như thế nào;
- sử dụng **axes-level functions** và **figure-level functions**;
- vẽ và diễn giải:
  - `lineplot`,
  - `scatterplot`,
  - `barplot`,
  - `countplot`,
  - `boxplot`,
  - `violinplot`,
  - `histplot`,
  - `kdeplot`,
  - `regplot`,
  - `heatmap`,
  - `pairplot`;
- sử dụng `hue`, `style`, `size`, `col`, `row` để trực quan hóa nhiều chiều;
- tùy biến theme, context, palette và kết hợp Seaborn với Matplotlib;
- xây dựng EDA/dashboard nhỏ cho bài toán Data Science, Business và Economics;
- viết nhận xét dựa trên bằng chứng trực quan.

## 0. Chuẩn bị môi trường

Nếu máy chưa có Seaborn:

```bash
pip install seaborn
```

hoặc:

```bash
conda install seaborn
```

Kiểm tra phiên bản và import thư viện.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)

sns.set_theme(style="whitegrid", context="notebook")

## 1. Tạo các bộ dữ liệu mẫu

Ta dùng 3 nhóm dữ liệu:

1. **Retail** — doanh thu, lợi nhuận, khu vực, sản phẩm.
2. **Customer** — hành vi khách hàng và churn.
3. **Economics** — inflation, unemployment, GDP growth theo quý.

Các dữ liệu đều dùng seed cố định để kết quả tái lập.

In [ ]:
rng = np.random.default_rng(42)

# -----------------------------
# Dataset 1: Retail
# -----------------------------
months = pd.date_range("2025-01-01", periods=12, freq="MS")
regions = ["North", "Central", "South", "Online"]
categories = ["Electronics", "Home", "Fashion"]

retail_rows = []
for m_idx, month in enumerate(months):
    trend = 1 + 0.025 * m_idx
    seasonal = 1 + 0.12 * np.sin(2 * np.pi * m_idx / 12)

    for region in regions:
        r_factor = {
            "North": 1.00,
            "Central": 0.85,
            "South": 1.12,
            "Online": 1.28,
        }[region]

        for category in categories:
            c_factor = {
                "Electronics": 1.35,
                "Home": 1.00,
                "Fashion": 0.88,
            }[category]

            revenue = (
                420 * trend * seasonal * r_factor * c_factor
                * rng.normal(1, 0.07)
            )

            margin = {
                "Electronics": 0.13,
                "Home": 0.20,
                "Fashion": 0.27,
            }[category] + rng.normal(0, 0.015)

            profit = revenue * margin
            orders = int(revenue * rng.uniform(1.8, 2.5))

            retail_rows.append(
                [month, region, category, revenue, profit, orders]
            )

retail = pd.DataFrame(
    retail_rows,
    columns=["Month", "Region", "Category", "Revenue", "Profit", "Orders"],
)

# Tạo một cú giảm mô phỏng ở Central vào tháng 8
mask = (retail["Month"] == "2025-08-01") & (retail["Region"] == "Central")
retail.loc[mask, ["Revenue", "Profit"]] *= [0.72, 0.70]

# -----------------------------
# Dataset 2: Customer churn
# -----------------------------
n = 1200
tenure = np.clip(rng.gamma(2.2, 11, n), 1, 72)
monthly_charge = np.clip(rng.normal(65, 18, n), 20, 120)
support_calls = rng.poisson(1.8, n)
contract = rng.choice(
    ["Monthly", "Annual", "Two-year"],
    size=n,
    p=[0.55, 0.30, 0.15],
)

logit = (
    -1.7
    + 0.020 * (monthly_charge - 60)
    - 0.035 * (tenure - 20)
    + 0.32 * support_calls
    + np.where(contract == "Monthly", 0.65, 0)
    + np.where(contract == "Two-year", -0.75, 0)
)

p_churn = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, p_churn)

customers = pd.DataFrame({
    "Tenure": tenure,
    "MonthlyCharge": monthly_charge,
    "SupportCalls": support_calls,
    "Contract": contract,
    "Churn": churn,
})

# -----------------------------
# Dataset 3: Economics
# -----------------------------
quarters = pd.period_range("2021Q1", "2025Q4", freq="Q")
t = np.arange(len(quarters))

inflation = (
    2.1
    + 4.8 * np.exp(-((t - 7) / 3.1) ** 2)
    + 0.25 * rng.normal(size=len(t))
)

unemployment = (
    5.4
    - 0.16 * t
    + 1.7 * np.exp(-((t - 13) / 2.4) ** 2)
    + 0.18 * rng.normal(size=len(t))
)

gdp_growth = (
    4.8
    + 0.7 * np.sin(t / 2)
    - 3.2 * np.exp(-((t - 13) / 2.0) ** 2)
    + 0.35 * rng.normal(size=len(t))
)

economy = pd.DataFrame({
    "Quarter": quarters.astype(str),
    "Inflation": inflation,
    "Unemployment": unemployment,
    "GDP_Growth": gdp_growth,
})

print("Retail shape:", retail.shape)
print("Customers shape:", customers.shape)
print("Economy shape:", economy.shape)

### Kiểm tra nhanh dữ liệu

Trước khi vẽ, luôn kiểm tra:

- mỗi hàng đại diện cho cái gì?
- biến nào là numerical?
- biến nào là categorical?
- có missing values không?
- dữ liệu có hợp lý không?

In [ ]:
display(retail.head())
display(customers.head())
display(economy.head())

print("\nMissing values:")
print("Retail:", retail.isna().sum().sum())
print("Customers:", customers.isna().sum().sum())
print("Economy:", economy.isna().sum().sum())

## 2. Seaborn và Matplotlib

Seaborn **không thay thế Matplotlib**.

Có thể hiểu:

- **Matplotlib** cung cấp nền tảng `Figure`, `Axes`, labels, annotations, layout...
- **Seaborn** cung cấp API thống kê và làm việc thuận tiện với DataFrame.

Ví dụ:

```python
fig, ax = plt.subplots()
sns.scatterplot(data=df, x="x", y="y", ax=ax)
ax.set_title("...")
```

Đây là cách kết hợp rất phổ biến trong Data Science.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    color=sns.color_palette("deep")[0],
    ax=ax,
)

ax.set_title("Seaborn plot inside a Matplotlib Axes")
ax.set_xlabel("Revenue")
ax.set_ylabel("Profit")

plt.tight_layout()
plt.show()

### Bài tập 1 — Seaborn + Matplotlib

Hãy sửa biểu đồ trên để:

1. phân nhóm theo `Category` bằng `hue`;
2. tăng kích thước Figure;
3. đổi title thành `"Revenue vs Profit by Product Category"`;
4. đặt legend bên ngoài nếu cần.

**Gợi ý:**

```python
sns.scatterplot(..., hue="Category", ax=ax)
ax.legend(...)
```

In [ ]:
# TODO - Bài tập 1

fig, ax = plt.subplots(figsize=(9, 5))

# sns.scatterplot(
#     data=retail,
#     x="Revenue",
#     y="Profit",
#     hue=...,
#     ax=ax,
# )

# ax.set_title(...)
# ax.legend(...)

plt.tight_layout()
plt.show()

## 3. Relational plots — quan hệ giữa các biến

Hai hàm quan trọng:

- `sns.scatterplot()` — quan hệ giữa hai biến số;
- `sns.lineplot()` — xu hướng theo biến có thứ tự hoặc thời gian.

Các tham số mạnh của Seaborn:

- `hue=` → màu;
- `style=` → kiểu marker/line;
- `size=` → kích thước;
- `data=` → DataFrame.

### 3.1 `scatterplot()` — quan hệ hai biến

Ví dụ: Revenue–Profit, phân nhóm theo Category.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    size="Orders",
    sizes=(30, 250),
    alpha=0.7,
    ax=ax,
)

ax.set_title("Revenue vs Profit")
plt.tight_layout()
plt.show()

### Diễn giải

Biểu đồ này mã hóa nhiều chiều:

- X: Revenue
- Y: Profit
- màu: Category
- kích thước: Orders

Đây là một ví dụ về **multidimensional visualization**.

### 3.2 `lineplot()` — xu hướng theo thời gian

Seaborn có thể tự tổng hợp dữ liệu khi nhiều quan sát có cùng X.

Ở đây ta tổng hợp trước để dễ kiểm soát kết quả.

In [ ]:
monthly_region = (
    retail.groupby(["Month", "Region"], as_index=False)["Revenue"]
    .sum()
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.lineplot(
    data=monthly_region,
    x="Month",
    y="Revenue",
    hue="Region",
    marker="o",
    ax=ax,
)

ax.set_title("Monthly Revenue by Region")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 2 — Relational plots

**A. Scatterplot**

Vẽ `Revenue` vs `Profit`:

- `hue="Region"`
- `style="Category"`
- `size="Orders"`

**B. Lineplot**

Vẽ tổng `Profit` theo `Month`, phân nhóm theo `Category`.

**Câu hỏi**

- Region nào xuất hiện nhiều ở vùng Revenue cao?
- Category nào có profit margin có vẻ cao hơn?
- Nhóm sản phẩm nào có xu hướng profit tăng rõ?

In [ ]:
# TODO - Bài tập 2A

fig, ax = plt.subplots(figsize=(10, 6))

# sns.scatterplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 2B

monthly_category_profit = (
    retail.groupby(["Month", "Category"], as_index=False)["Profit"]
    .sum()
)

fig, ax = plt.subplots(figsize=(11, 5))

# sns.lineplot(...)

plt.tight_layout()
plt.show()

## 4. Categorical plots — so sánh giữa các nhóm

Các hàm phổ biến:

- `barplot()` — ước lượng trung tâm theo nhóm;
- `countplot()` — số lượng quan sát;
- `boxplot()` — median, quartiles, spread, outliers;
- `violinplot()` — hình dạng phân phối theo nhóm;
- `stripplot()` / `swarmplot()` — hiển thị từng quan sát.

> Lưu ý: `barplot()` không đơn giản chỉ là vẽ tổng. Mặc định nó tính một estimator (thường là mean).

### 4.1 `barplot()` — so sánh giá trị trung bình

Ví dụ: Revenue trung bình theo Region.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=retail,
    x="Region",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=ax,
)

ax.set_title("Average Revenue by Region")
plt.tight_layout()
plt.show()

### 4.2 `countplot()` — đếm quan sát theo nhóm

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.countplot(
    data=customers,
    x="Contract",
    hue="Churn",
    ax=ax,
)

ax.set_title("Customer Count by Contract and Churn")
plt.tight_layout()
plt.show()

### 4.3 `boxplot()` — phân phối theo nhóm

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=customers,
    x="Contract",
    y="MonthlyCharge",
    hue="Churn",
    ax=ax,
)

ax.set_title("Monthly Charge by Contract and Churn")
plt.tight_layout()
plt.show()

### 4.4 `violinplot()` — nhìn thêm hình dạng phân phối

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=customers,
    x="Contract",
    y="Tenure",
    hue="Churn",
    split=True,
    inner="quart",
    ax=ax,
)

ax.set_title("Tenure Distribution by Contract and Churn")
plt.tight_layout()
plt.show()

### Bài tập 3 — Categorical plots

1. Dùng `barplot()` để so sánh **Profit trung bình theo Category**, phân nhóm thêm theo `Region`.
2. Dùng `countplot()` để xem số khách hàng churn theo `Contract`.
3. Dùng `boxplot()` để so sánh `Tenure` giữa `Churn=0` và `Churn=1`.
4. Viết 2–3 nhận xét.

**Chú ý:** Khi dùng `barplot`, hãy nói rõ bạn đang hiển thị **mean**, không phải total.

In [ ]:
# TODO - Bài tập 3.1
fig, ax = plt.subplots(figsize=(10, 5))

# sns.barplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 3.2
fig, ax = plt.subplots(figsize=(7, 5))

# sns.countplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 3.3
fig, ax = plt.subplots(figsize=(7, 5))

# sns.boxplot(...)

plt.tight_layout()
plt.show()

## 5. Distribution plots — khám phá phân phối

Các hàm quan trọng:

- `histplot()` — histogram;
- `kdeplot()` — Kernel Density Estimate;
- `ecdfplot()` — empirical cumulative distribution.

Histogram cho thấy **số lượng/tần suất theo khoảng**.  
KDE cung cấp một đường ước lượng mượt của mật độ.

### 5.1 `histplot()`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=customers,
    x="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    bins=25,
    kde=True,
    ax=ax,
)

ax.set_title("Distribution of Monthly Charge")
plt.tight_layout()
plt.show()

### 5.2 Histogram theo nhóm với `hue`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=25,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.35,
    ax=ax,
)

ax.set_title("Tenure Distribution by Churn")
plt.tight_layout()
plt.show()

### 5.3 `kdeplot()`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.kdeplot(
    data=customers,
    x="MonthlyCharge",
    hue="Churn",
    fill=True,
    common_norm=False,
    alpha=0.35,
    ax=ax,
)

ax.set_title("Monthly Charge Density by Churn")
plt.tight_layout()
plt.show()

### Bài tập 4 — Distribution

1. Vẽ histogram `SupportCalls`, phân nhóm theo `Churn`.
2. Vẽ KDE cho `Tenure`, phân nhóm theo `Contract`.
3. So sánh histogram và KDE:
   - cái nào cho count rõ hơn?
   - cái nào cho shape rõ hơn?

In [ ]:
# TODO - Bài tập 4.1
fig, ax = plt.subplots(figsize=(8, 5))

# sns.histplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 4.2
fig, ax = plt.subplots(figsize=(9, 5))

# sns.kdeplot(...)

plt.tight_layout()
plt.show()

## 6. Regression plots — quan hệ và đường hồi quy

Hai hàm thường dùng:

- `sns.regplot()` — axes-level;
- `sns.lmplot()` — figure-level.

`regplot()` phù hợp khi bạn muốn đặt biểu đồ vào `Axes` hiện có.

> Đường hồi quy giúp mô tả xu hướng. Nó **không tự động chứng minh quan hệ nhân quả**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(
    data=retail,
    x="Revenue",
    y="Profit",
    color=sns.color_palette("deep")[0],
    scatter_kws={"alpha": 0.5},
    line_kws={"linewidth": 2},
    ax=ax,
)

ax.set_title("Regression View: Revenue vs Profit")
plt.tight_layout()
plt.show()

### `lmplot()` với phân nhóm

In [ ]:
g = sns.lmplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    height=5,
    aspect=1.4,
    scatter_kws={"alpha": 0.5},
)

g.fig.suptitle("Revenue vs Profit by Category", y=1.03)
plt.show()

### Bài tập 5 — Regression plots

Dùng dữ liệu `economy`:

1. vẽ `Unemployment` vs `Inflation` bằng `regplot()`;
2. thêm title `"Inflation vs Unemployment"`;
3. tính correlation;
4. trả lời: có nên kết luận thất nghiệp gây ra lạm phát không?

In [ ]:
# TODO - Bài tập 5

corr = economy["Unemployment"].corr(economy["Inflation"])
print("Correlation:", round(corr, 3))

fig, ax = plt.subplots(figsize=(8, 5))

# sns.regplot(...)

plt.tight_layout()
plt.show()

## 7. Matrix plots — `heatmap()`

Heatmap thường dùng để:

- biểu diễn correlation matrix;
- missing-value matrix;
- pivot table;
- confusion matrix.

Ví dụ: correlation giữa các biến numerical trong dữ liệu Customer.

In [ ]:
corr = customers[
    ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
].corr()

fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
)

ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### Heatmap từ pivot table

Ta có thể biến dữ liệu Business thành ma trận trước khi vẽ.

In [ ]:
pivot_revenue = retail.pivot_table(
    index="Region",
    columns="Category",
    values="Revenue",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    pivot_revenue,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=ax,
)

ax.set_title("Average Revenue: Region × Category")
plt.tight_layout()
plt.show()

### Bài tập 6 — Heatmap

1. Tạo correlation heatmap cho:
   - `Revenue`,
   - `Profit`,
   - `Orders`.
2. Tạo pivot table:
   - rows = `Category`
   - columns = `Region`
   - value = `Profit`
   - aggfunc = `"mean"`
3. Vẽ heatmap từ pivot table.
4. Region–Category nào có profit trung bình cao nhất?

In [ ]:
# TODO - Bài tập 6.1

In [ ]:
# TODO - Bài tập 6.2

## 8. `pairplot()` — khám phá nhiều biến cùng lúc

`pairplot()` tạo:

- scatterplots cho từng cặp biến;
- distribution plots trên đường chéo.

Nó rất hữu ích ở giai đoạn **EDA ban đầu**.

In [ ]:
sample_customers = customers.sample(400, random_state=42)

g = sns.pairplot(
    sample_customers[
        ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
    ],
    hue="Churn",
    corner=True,
)

g.fig.suptitle("Pairwise EDA for Customer Churn", y=1.02)
plt.show()

### Bài tập 7 — Pairplot

Tạo `pairplot()` cho Retail với:

- `Revenue`
- `Profit`
- `Orders`
- `Category`

Yêu cầu:

- `hue="Category"`
- `corner=True`

Sau đó tìm ít nhất **2 pattern** có thể quan sát được.

In [ ]:
# TODO - Bài tập 7

## 9. Axes-level và Figure-level APIs

### Axes-level

Ví dụ:

- `scatterplot`
- `lineplot`
- `barplot`
- `boxplot`
- `histplot`
- `regplot`
- `heatmap`

Có thể truyền `ax=`.

### Figure-level

Ví dụ:

- `relplot`
- `catplot`
- `displot`
- `lmplot`

Chúng tự quản lý Figure và hỗ trợ faceting bằng `row=` / `col=`.

### `relplot()` — scatter nhiều panel

In [ ]:
g = sns.relplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    col="Region",
    col_wrap=2,
    height=4,
)

g.fig.suptitle("Revenue vs Profit by Region", y=1.02)
plt.show()

### `catplot()` — categorical faceting

In [ ]:
g = sns.catplot(
    data=customers,
    x="Contract",
    y="Tenure",
    col="Churn",
    kind="box",
    color=sns.color_palette("deep")[0],
    height=4,
    aspect=1.1,
)

g.fig.suptitle("Tenure by Contract, Faceted by Churn", y=1.03)
plt.show()

### `displot()` — distribution faceting

In [ ]:
g = sns.displot(
    data=customers,
    x="MonthlyCharge",
    col="Contract",
    hue="Churn",
    kind="hist",
    bins=20,
    common_norm=False,
    height=4,
)

g.fig.suptitle("Monthly Charge Distribution by Contract", y=1.03)
plt.show()

### Bài tập 8 — Figure-level plots

1. Dùng `relplot()` để vẽ Revenue–Profit:
   - `hue="Category"`
   - `col="Region"`
2. Dùng `catplot(kind="box")` để vẽ Tenure theo Contract, `col="Churn"`.
3. Giải thích khi nào Figure-level thuận tiện hơn Axes-level.

In [ ]:
# TODO - Bài tập 8.1

In [ ]:
# TODO - Bài tập 8.2

## 10. Theme, style, context và palette

Seaborn hỗ trợ:

```python
sns.set_theme(...)
sns.set_style(...)
sns.set_context(...)
sns.color_palette(...)
```

Một số style:

- `"whitegrid"`
- `"darkgrid"`
- `"white"`
- `"dark"`
- `"ticks"`

Một số context:

- `"paper"`
- `"notebook"`
- `"talk"`
- `"poster"`

In [ ]:
sns.set_theme(style="ticks", context="talk")

fig, ax = plt.subplots(figsize=(9, 5))

sns.lineplot(
    data=monthly_region,
    x="Month",
    y="Revenue",
    hue="Region",
    marker="o",
    ax=ax,
)

sns.despine()
ax.set_title("Same Data, Different Seaborn Theme")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Trả về mặc định cho notebook
sns.set_theme(style="whitegrid", context="notebook")

### Bài tập 9 — Style

Tạo cùng một biểu đồ dưới 2 style khác nhau:

- `whitegrid`
- `ticks`

Viết nhận xét:

- style nào phù hợp báo cáo Business?
- style nào phù hợp slide thuyết trình?
- vì sao đây là quyết định thiết kế chứ không phải quyết định thống kê?

In [ ]:
# TODO - Bài tập 9

## 11. Subplots — kết hợp nhiều biểu đồ Seaborn

Ta vẫn dùng `plt.subplots()` của Matplotlib, rồi truyền từng `Axes` cho Seaborn.

Đây là kỹ thuật quan trọng khi tạo dashboard.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

sns.barplot(
    data=retail,
    x="Region",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0],
)
axes[0].set_title("Average Revenue")

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Revenue vs Profit")

sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=20,
    element="step",
    ax=axes[2],
)
axes[2].set_title("Customer Tenure")

fig.suptitle("Mini Analytics Dashboard", fontsize=16, y=1.03)
plt.tight_layout()
plt.show()

### Bài tập 10 — Dashboard 2×2

Tạo dashboard gồm:

1. `lineplot`: Revenue theo Month;
2. `barplot`: Profit theo Category;
3. `boxplot`: MonthlyCharge theo Churn;
4. `heatmap`: correlation của customer numerical variables.

Yêu cầu:

- `figsize=(14, 9)`
- title cho từng biểu đồ;
- `fig.suptitle(...)`;
- `plt.tight_layout()`.

In [ ]:
# TODO - Bài tập 10

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# TODO: axes[0, 0]
# TODO: axes[0, 1]
# TODO: axes[1, 0]
# TODO: axes[1, 1]

fig.suptitle("Seaborn Analytics Dashboard", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 12. Annotation — khi Seaborn cần Matplotlib

Seaborn tạo plot nhanh, nhưng annotation thường dùng API Matplotlib:

```python
ax.annotate(...)
ax.axhline(...)
ax.axvline(...)
ax.axvspan(...)
```

Ví dụ: đánh dấu quý có GDP Growth thấp nhất.

In [ ]:
min_idx = economy["GDP_Growth"].idxmin()
x_min = min_idx
y_min = economy.loc[min_idx, "GDP_Growth"]

fig, ax = plt.subplots(figsize=(11, 5))

sns.lineplot(
    data=economy,
    x="Quarter",
    y="GDP_Growth",
    color=sns.color_palette("deep")[0],
    marker="o",
    ax=ax,
)

ax.axhline(0, linestyle="--", linewidth=1)

ax.annotate(
    f'Min: {economy.loc[min_idx, "Quarter"]}\n{y_min:.2f}',
    xy=(x_min, y_min),
    xytext=(x_min - 4, y_min + 1.1),
    arrowprops={"arrowstyle": "->"},
)

ax.set_title("GDP Growth with Annotation")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 11 — Data storytelling

Trên lineplot tổng Revenue theo Month:

1. tìm tháng Revenue cao nhất;
2. dùng `annotate()` đánh dấu;
3. thêm đường mean bằng `ax.axhline()`;
4. đổi title thành một **message title**, không chỉ là tên biểu đồ.

Ví dụ:

> `"Revenue Peaks at Year-End After a Mid-Year Slowdown"`

In [ ]:
# TODO - Bài tập 11

## 13. Ứng dụng Data Science — EDA cho Customer Churn

Một workflow EDA trực quan cơ bản:

1. Target distribution
2. Numerical distributions
3. Numerical vs target
4. Categorical vs target
5. Correlations
6. Multivariate relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.countplot(
    data=customers,
    x="Churn",
    color=sns.color_palette("deep")[0],
    ax=axes[0, 0],
)
axes[0, 0].set_title("Target Distribution")

sns.boxplot(
    data=customers,
    x="Churn",
    y="Tenure",
    color=sns.color_palette("deep")[0],
    ax=axes[0, 1],
)
axes[0, 1].set_title("Tenure vs Churn")

sns.boxplot(
    data=customers,
    x="Churn",
    y="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    ax=axes[1, 0],
)
axes[1, 0].set_title("Monthly Charge vs Churn")

customer_corr = customers[
    ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
].corr()

sns.heatmap(
    customer_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Correlation Matrix")

fig.suptitle("Customer Churn — EDA Overview", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Câu hỏi Data Science

Dựa trên các biểu đồ:

1. Target có mất cân bằng nghiêm trọng không?
2. `Tenure` có vẻ hữu ích cho dự đoán churn không?
3. `MonthlyCharge` có khác giữa hai nhóm không?
4. `SupportCalls` có tương quan với churn không?
5. Có thể chọn feature chỉ dựa vào correlation không?

Viết câu trả lời bằng ngôn ngữ phân tích, tránh khẳng định nhân quả.

## 14. Ứng dụng Business — Sales & Profitability

Mục tiêu: đưa ra một dashboard cho manager.

Ta muốn nhìn:

- trend,
- comparison,
- profitability,
- region × category.

In [ ]:
monthly_total = retail.groupby("Month", as_index=False)["Revenue"].sum()
region_profit = retail.groupby("Region", as_index=False)["Profit"].sum()
profit_matrix = retail.pivot_table(
    index="Region",
    columns="Category",
    values="Profit",
    aggfunc="sum",
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.lineplot(
    data=monthly_total,
    x="Month",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    marker="o",
    ax=axes[0, 0],
)
axes[0, 0].set_title("Monthly Revenue")
axes[0, 0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=region_profit,
    x="Region",
    y="Profit",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Total Profit by Region")

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    size="Orders",
    sizes=(20, 180),
    alpha=0.6,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Revenue vs Profit")

sns.heatmap(
    profit_matrix,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=axes[1, 1],
)
axes[1, 1].set_title("Profit: Region × Category")

fig.suptitle("Retail Business Performance", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Câu hỏi Business

Viết **3–5 insights** theo cấu trúc:

- **Observation**
- **Evidence**
- **Business implication**

Ví dụ:

> Online có doanh thu cao nhưng cần xem đồng thời profit để tránh kết luận rằng doanh thu cao đồng nghĩa hiệu quả cao.

## 15. Ứng dụng Economics — Inflation, Unemployment, GDP Growth

Seaborn hỗ trợ tốt việc:

- vẽ time series;
- so sánh hai macro indicators;
- kết hợp regression view và correlation.

Nhưng trực quan hóa không thay thế mô hình kinh tế lượng.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

sns.lineplot(
    data=economy,
    x="Quarter",
    y="Inflation",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Inflation")

sns.lineplot(
    data=economy,
    x="Quarter",
    y="Unemployment",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Unemployment")

sns.lineplot(
    data=economy,
    x="Quarter",
    y="GDP_Growth",
    marker="o",
    ax=axes[2],
)
axes[2].axhline(0, linestyle="--", linewidth=1)
axes[2].set_title("GDP Growth")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 12 — Economics visualization

Tạo Figure gồm 2 biểu đồ:

**Trái**
- `regplot`: Unemployment vs Inflation

**Phải**
- `scatterplot`: GDP Growth vs Inflation
- dùng `size="Unemployment"` bằng cách truyền column tương ứng

Sau đó viết 4–6 câu nhận xét và nêu rõ:

> Correlation/visual association ≠ causation.

In [ ]:
# TODO - Bài tập 12
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# TODO: regplot bên trái
# TODO: scatterplot bên phải

plt.tight_layout()
plt.show()

## 16. Xuất biểu đồ

Dù plot được tạo bằng Seaborn, Figure vẫn là Matplotlib Figure.

Do đó có thể dùng:

```python
fig.savefig(...)
```

In [ ]:
output_dir = Path("seaborn_outputs")
output_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    ax=ax,
)

ax.set_title("Revenue vs Profit")

output_file = output_dir / "revenue_profit.png"
fig.savefig(output_file, dpi=180, bbox_inches="tight")

plt.show()

print("Saved to:", output_file.resolve())

## 17. Viết hàm vẽ tái sử dụng

Một nguyên tắc tốt là viết hàm nhận `ax` thay vì luôn tự tạo Figure.

Nhờ đó hàm có thể dùng trong dashboard.

In [ ]:
def plot_group_boxplot(data, x, y, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    sns.boxplot(
        data=data,
        x=x,
        y=y,
        color=sns.color_palette("deep")[0],
        ax=ax,
    )

    ax.set_title(title or f"{y} by {x}")
    return ax


fig, ax = plt.subplots(figsize=(8, 5))
plot_group_boxplot(
    customers,
    x="Contract",
    y="MonthlyCharge",
    ax=ax,
    title="Monthly Charge by Contract",
)
plt.tight_layout()
plt.show()

### Bài tập 13 — Mở rộng hàm

Sửa hàm trên để nhận thêm:

- `hue=None`
- `palette=None`

Sau đó dùng hàm để vẽ:

- `Tenure` theo `Contract`
- `hue="Churn"`

In [ ]:
# TODO - Bài tập 13

## 18. Bài thực hành có lời giải — EDA nhanh cho Customer Churn

### Đề bài

Trong vai Data Analyst, hãy tạo một báo cáo trực quan ngắn để trả lời:

1. Churn rate khoảng bao nhiêu?
2. Contract type nào có churn rate cao hơn?
3. Khách hàng churn có tenure khác không?
4. Monthly charge có liên hệ với churn không?
5. Feature nào nên được xem xét ở bước modeling?

Phần dưới là **một lời giải mẫu**.

In [ ]:
# 1. Tạo churn rate theo contract
churn_by_contract = (
    customers.groupby("Contract", as_index=False)["Churn"]
    .mean()
    .sort_values("Churn", ascending=False)
)

display(churn_by_contract)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) Churn rate theo contract
sns.barplot(
    data=churn_by_contract,
    x="Contract",
    y="Churn",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Churn Rate by Contract")
axes[0, 0].set_ylabel("Churn Rate")

# (2) Tenure distribution
sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=24,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Tenure Distribution")

# (3) Monthly Charge
sns.boxplot(
    data=customers,
    x="Churn",
    y="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    ax=axes[1, 0],
)
axes[1, 0].set_title("Monthly Charge by Churn")

# (4) Multivariate view
sns.scatterplot(
    data=customers.sample(500, random_state=42),
    x="Tenure",
    y="MonthlyCharge",
    hue="Churn",
    size="SupportCalls",
    sizes=(20, 150),
    alpha=0.55,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Customer Profile")

fig.suptitle("Customer Churn — Guided EDA", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Gợi ý diễn giải lời giải

Một báo cáo tốt không chỉ mô tả biểu đồ mà phải kết nối với câu hỏi phân tích.

Ví dụ:

- Nếu nhóm Monthly contract có churn rate cao hơn, `Contract` là biến đáng kiểm tra trong modeling.
- Nếu churn tập trung ở tenure thấp, doanh nghiệp nên chú ý giai đoạn onboarding/early lifecycle.
- Nếu MonthlyCharge của nhóm churn cao hơn, giá có thể là một tín hiệu liên quan — nhưng chưa thể kết luận là nguyên nhân.
- `SupportCalls` có thể phản ánh friction/service issues và nên được kiểm tra thêm.
- Cần tiếp tục đánh giá bằng mô hình, validation và domain knowledge.

## 19. Bài tập tổng hợp tự làm

### PROJECT — Data Visualization Report bằng Seaborn

Chọn **một trong ba bối cảnh**:

### A. Business Analytics
Dùng `retail`.

### B. Data Science
Dùng `customers`.

### C. Economics
Dùng `economy`.

### Sản phẩm cần nộp

Một notebook/report gồm ít nhất:

1. **1 relational plot**
2. **1 categorical plot**
3. **1 distribution plot**
4. **1 heatmap hoặc pairplot**
5. **1 dashboard ít nhất 2×2**
6. **ít nhất 1 annotation**
7. **5 insights**
8. **2 đề xuất ra quyết định / bước phân tích tiếp theo**

### Ràng buộc

- Mỗi hình phải có title và labels phù hợp.
- Không dùng biểu đồ chỉ vì “đẹp”.
- Phải giải thích tại sao chọn loại biểu đồ đó.
- Không suy diễn correlation thành causation.

### Khung làm bài

#### Bước 1 — Câu hỏi phân tích

Viết 3–5 câu hỏi trước khi vẽ.

> 1. ...
>
> 2. ...
>
> 3. ...

#### Bước 2 — Univariate analysis

> ...

#### Bước 3 — Bivariate / multivariate analysis

> ...

#### Bước 4 — Dashboard

> ...

#### Bước 5 — Insights

> 1. ...
>
> 2. ...
>
> 3. ...
>
> 4. ...
>
> 5. ...

#### Bước 6 — Recommendation / next steps

> ...

In [ ]:
# TODO - PROJECT TỰ LÀM

# Chọn dataset:
# df = retail.copy()
# df = customers.copy()
# df = economy.copy()

# Bắt đầu phân tích tại đây.

## 20. Cheat sheet lệnh Seaborn

| Mục đích | Hàm |
|---|---|
| Scatter | `sns.scatterplot()` |
| Line | `sns.lineplot()` |
| Bar mean/estimator | `sns.barplot()` |
| Count | `sns.countplot()` |
| Boxplot | `sns.boxplot()` |
| Violin | `sns.violinplot()` |
| Histogram | `sns.histplot()` |
| KDE | `sns.kdeplot()` |
| ECDF | `sns.ecdfplot()` |
| Regression | `sns.regplot()` |
| Regression + facet | `sns.lmplot()` |
| Heatmap | `sns.heatmap()` |
| Pairwise EDA | `sns.pairplot()` |
| Relational facet | `sns.relplot()` |
| Categorical facet | `sns.catplot()` |
| Distribution facet | `sns.displot()` |
| Theme | `sns.set_theme()` |
| Style | `sns.set_style()` |
| Context | `sns.set_context()` |
| Remove spines | `sns.despine()` |

### Semantic mappings

| Tham số | Ý nghĩa |
|---|---|
| `x` | biến trục X |
| `y` | biến trục Y |
| `hue` | phân nhóm bằng màu |
| `style` | phân nhóm bằng marker/line style |
| `size` | phân nhóm bằng kích thước |
| `row` | facet theo hàng |
| `col` | facet theo cột |
| `data` | DataFrame |
| `ax` | Matplotlib Axes |

## Kết thúc

Sau notebook này, người học nên phân biệt được:

### Khi nào dùng Matplotlib?
- cần kiểm soát layout/annotation chi tiết;
- cần custom graphics;
- cần xây Figure phức tạp.

### Khi nào dùng Seaborn?
- làm việc với DataFrame;
- cần statistical visualization nhanh;
- cần `hue`, `style`, `size`, faceting;
- cần EDA hiệu quả với ít code.

### Thực tế
Hai thư viện thường được **dùng cùng nhau**:

```python
fig, ax = plt.subplots()
sns.someplot(..., ax=ax)
ax.set_title(...)
ax.annotate(...)
plt.show()
```

Mục tiêu cuối cùng không phải “biết nhiều hàm vẽ”, mà là:

> **chọn đúng biểu đồ → đọc đúng dữ liệu → truyền đạt đúng insight.**